# D163 — MySQL Analytics Queries on Olist Order Items

This notebook uses only the `olist_order_items` table created in D161. One row represents one item inside an order.

The table is useful for learning because it contains:

- identifiers: `order_id`, `product_id`, and `seller_id`;
- a sequence number: `order_item_id`;
- a date: `shipping_limit_date`; and
- numeric measures: `price` and `freight_value`.

All examples are read-only. They do not change the imported data.

## 1. Connect to MySQL

Connection values come from environment variables. The defaults match the local classroom setup.

In [ ]:
import os
import mysql.connector

connection = mysql.connector.connect(
    host=os.environ.get('MYSQL_HOSTNAME', '127.0.0.1'),
    port=int(os.environ.get('MYSQL_PORT', '3306')),
    user=os.environ.get('MYSQL_USERNAME', 'root'),
    password=os.environ.get('MYSQL_PASSWORD', 'root'),
    database=os.environ.get('MYSQL_DATABASE', 'olist_import_lab'),
)
print('Connected:', connection.is_connected())
print('MySQL version:', connection.server_info)

### Query helper

A **cursor** is the connector object that sends SQL to MySQL and receives rows. The helper prints results as an aligned table. `max_rows` prevents a large result from filling the notebook.

In [ ]:
def execute_sql(sql, params=None, max_rows=25):
    cursor = connection.cursor()
    try:
        cursor.execute(sql, params or ())
        columns = [item[0] for item in cursor.description]
        rows = cursor.fetchmany(max_rows + 1)
        more_rows = len(rows) > max_rows
        rows = rows[:max_rows]

        text_rows = [[('NULL' if value is None else str(value)) for value in row]
                     for row in rows]
        widths = [len(str(column)) for column in columns]
        for row in text_rows:
            widths = [max(width, len(value)) for width, value in zip(widths, row)]

        print(' | '.join(str(c).ljust(w) for c, w in zip(columns, widths)))
        print('-+-'.join('-' * w for w in widths))
        for row in text_rows:
            print(' | '.join(value.ljust(w) for value, w in zip(row, widths)))
        if more_rows:
            print(f'... showing the first {max_rows} rows')
        return rows
    finally:
        cursor.close()

## 2. Look at the table

`LIMIT` restricts the number of returned rows. It is useful when checking a large table. Without `ORDER BY`, MySQL does not promise which rows will appear first.

In [ ]:
execute_sql("""
SELECT *
FROM olist_order_items
ORDER BY order_id, order_item_id
LIMIT 5
""")

## 3. Create a calculated value

An **expression** calculates a value for each row. Here, the amount charged for an item is its price plus freight. `AS item_total` gives the result a readable alias.

In [ ]:
execute_sql("""
SELECT order_id, order_item_id, price, freight_value,
       price + freight_value AS item_total
FROM olist_order_items
ORDER BY item_total DESC
LIMIT 10
""")

## 4. Filter rows with `WHERE`

`WHERE` removes rows before grouping or calculation. The query below finds items priced from 500 through 1,000 BRL with freight below 50 BRL. `BETWEEN` includes both end values.

In [ ]:
execute_sql("""
SELECT order_id, product_id, seller_id, price, freight_value
FROM olist_order_items
WHERE price BETWEEN 500 AND 1000
  AND freight_value < 50
ORDER BY price DESC, freight_value ASC
LIMIT 10
""")

## 5. `COUNT` and `COUNT(DISTINCT ...)`

`COUNT(*)` counts rows. `COUNT(DISTINCT column)` counts different non-null values. Item rows, orders, products, and sellers are different business counts.

In [ ]:
execute_sql("""
SELECT COUNT(*) AS item_rows,
       COUNT(DISTINCT order_id) AS orders,
       COUNT(DISTINCT product_id) AS products,
       COUNT(DISTINCT seller_id) AS sellers
FROM olist_order_items
""")

## 6. Basic aggregate functions

An **aggregate** combines many rows into one result.

- `MIN` returns the smallest value.
- `MAX` returns the largest value.
- `SUM` adds the values.
- `AVG` returns the arithmetic mean. Mean and average mean the same thing here.
- `ROUND(value, 2)` displays two decimal places.

In [ ]:
execute_sql("""
SELECT MIN(price) AS minimum_price,
       MAX(price) AS maximum_price,
       ROUND(AVG(price), 2) AS average_price,
       ROUND(SUM(price), 2) AS total_item_value,
       ROUND(MIN(freight_value), 2) AS minimum_freight,
       ROUND(MAX(freight_value), 2) AS maximum_freight,
       ROUND(AVG(freight_value), 2) AS average_freight,
       ROUND(SUM(freight_value), 2) AS total_freight
FROM olist_order_items
""")

## 7. Group rows with `GROUP BY`

`GROUP BY` makes one result row for each distinct grouping value. This query produces one row per seller. `ORDER BY ... DESC` places the largest item value first.

In [ ]:
execute_sql("""
SELECT seller_id,
       COUNT(*) AS items_sold,
       COUNT(DISTINCT order_id) AS orders_served,
       ROUND(SUM(price), 2) AS item_value,
       ROUND(AVG(price), 2) AS average_item_price
FROM olist_order_items
GROUP BY seller_id
ORDER BY item_value DESC
LIMIT 10
""")

## 8. Filter groups with `HAVING`

`WHERE` filters source rows before grouping. `HAVING` filters the completed groups after `COUNT`, `SUM`, or another aggregate has been calculated. Here, a seller must have at least 100 item rows and more than 100,000 BRL in item value.

In [ ]:
execute_sql("""
SELECT seller_id, COUNT(*) AS items_sold, ROUND(SUM(price), 2) AS item_value
FROM olist_order_items
GROUP BY seller_id
HAVING COUNT(*) >= 100
   AND SUM(price) > 100000
ORDER BY item_value DESC
""")

## 9. Use `WHERE` and `HAVING` together

First, `WHERE` keeps only items shipped during 2018. Then `GROUP BY` creates seller groups. Finally, `HAVING` keeps sellers with at least 500 items in that filtered period. A half-open date range (`>= 2018-01-01` and `< 2019-01-01`) works safely even when the column contains times.

In [ ]:
execute_sql("""
SELECT seller_id, COUNT(*) AS items, ROUND(SUM(price), 2) AS item_value
FROM olist_order_items
WHERE shipping_limit_date >= '2018-01-01'
  AND shipping_limit_date <  '2019-01-01'
GROUP BY seller_id
HAVING COUNT(*) >= 500
ORDER BY items DESC, seller_id
""")

## 10. Group by month

`DATE_FORMAT(date, '%Y-%m')` creates a year-month label. Grouping by only the month number would mix January 2017 with January 2018.

In [ ]:
execute_sql("""
SELECT DATE_FORMAT(shipping_limit_date, '%Y-%m') AS shipping_month,
       COUNT(*) AS items,
       ROUND(SUM(price), 2) AS item_value,
       ROUND(SUM(freight_value), 2) AS freight_value
FROM olist_order_items
GROUP BY DATE_FORMAT(shipping_limit_date, '%Y-%m')
ORDER BY shipping_month
""")

## 11. Conditional aggregation

**Conditional aggregation** counts or adds only rows that meet a condition. `CASE` returns 1 for a matching row and 0 otherwise; `SUM` adds those flags. This produces several price bands in one query.

In [ ]:
execute_sql("""
SELECT COUNT(*) AS all_items,
       SUM(CASE WHEN price < 50 THEN 1 ELSE 0 END) AS under_50,
       SUM(CASE WHEN price >= 50 AND price < 200 THEN 1 ELSE 0 END) AS from_50_to_199,
       SUM(CASE WHEN price >= 200 AND price < 1000 THEN 1 ELSE 0 END) AS from_200_to_999,
       SUM(CASE WHEN price >= 1000 THEN 1 ELSE 0 END) AS at_least_1000
FROM olist_order_items
""")

## 12. Percentage and safe division

Freight percentage is `freight / price × 100`. `NULLIF(price, 0)` prevents division by zero by returning null when price is zero. `AVG` ignores null values.

In [ ]:
execute_sql("""
SELECT ROUND(100 * SUM(freight_value) / NULLIF(SUM(price), 0), 2)
           AS freight_as_percent_of_item_value,
       ROUND(AVG(100 * freight_value / NULLIF(price, 0)), 2)
           AS average_row_freight_percent
FROM olist_order_items
""")

## 13. Find orders with several items

The composite primary key tells us that `order_item_id` starts again for each order. Grouping by `order_id` shows orders containing several item rows.

In [ ]:
execute_sql("""
SELECT order_id, COUNT(*) AS item_count,
       COUNT(DISTINCT product_id) AS different_products,
       ROUND(SUM(price + freight_value), 2) AS order_item_total
FROM olist_order_items
GROUP BY order_id
HAVING COUNT(*) >= 5
ORDER BY item_count DESC, order_item_total DESC
LIMIT 10
""")

## 14. Subquery: compare with the overall average

A **subquery** is a query inside another query. The inner query calculates one overall average. The outer query returns items priced at least ten times above that average.

In [ ]:
execute_sql("""
SELECT order_id, product_id, price
FROM olist_order_items
WHERE price >= (SELECT AVG(price) * 10 FROM olist_order_items)
ORDER BY price DESC
LIMIT 10
""")

## 15. CTE: name a temporary result

A **Common Table Expression**, or CTE, is a named query result used by the statement that follows it. It starts with `WITH`. This makes a multi-step query easier to read.

In [ ]:
execute_sql("""
WITH seller_totals AS (
    SELECT seller_id, COUNT(*) AS items, SUM(price) AS item_value
    FROM olist_order_items
    GROUP BY seller_id
)
SELECT seller_id, items, ROUND(item_value, 2) AS item_value
FROM seller_totals
WHERE item_value >= 150000
ORDER BY item_value DESC
""")

## 16. Logical query order

SQL is written in one order but is understood logically in another:

1. `FROM` chooses the table.
2. `WHERE` filters source rows.
3. `GROUP BY` forms groups.
4. aggregate functions calculate group values.
5. `HAVING` filters groups.
6. `SELECT` produces the requested columns.
7. `ORDER BY` sorts the result.
8. `LIMIT` restricts the returned rows.

This explains why an aggregate condition belongs in `HAVING` instead of `WHERE`.

## 17. Summary

The notebook used one table to cover the most common MySQL analysis tools:

- `WHERE`, `BETWEEN`, `AND`, aliases, calculated columns, and safe division;
- `COUNT`, `COUNT DISTINCT`, `MIN`, `MAX`, `SUM`, `AVG`, and `ROUND`;
- `GROUP BY`, `HAVING`, `ORDER BY`, and `LIMIT`;
- date grouping and conditional aggregation with `CASE`; and
- subqueries and CTEs.

Window functions begin in D164 and are applied to Olist data in D169.

In [ ]:
if connection.is_connected():
    connection.close()
print('MySQL connection closed.')